# ACE-Step 1.5XL captioning (generated)

**Before running:** Session options -> Accelerator = **GPU T4 x2**, Internet = **On**.
The kernel loads with `device_map="balanced"` (10 GiB per GPU) and pip-installs a
transformers fork from GitHub, so neither can be skipped.

Then **Run All**. Results land in `/kaggle/working/captions_out.json`:

```json
{"results": [{"file": "Song.mp3", "caption": "..."}]}
```

Captions follow the ACE-Step 1.5XL annotation schema (front-loaded 5-12 keywords,
a mandatory vocal descriptor, 2-3 sentences of flow, no BPM/key/time signature).
An over-long caption is trimmed back to the schema.

Regenerate with `.venv/bin/python scripts/build_caption_notebook.py` — do not
hand-edit the cell.

In [ ]:
"""ACE-Step audio captioner — Kaggle kernel.

Based on the working ACE-Step captioner notebook. Reads audio from a mounted
Kaggle dataset, runs the Qwen2.5-Omni captioner with the proper chat template,
writes ``/kaggle/working/captions_out.json``::

    {"results": [{"file": "<original filename>", "caption": "..."}, ...]}

Placeholders substituted by the app at push time:
  /kaggle/input/acestep-audio  -> /kaggle/input/<audio-dataset-name>
  "Annotate this audio clip as ONE caption in the exact schema above. Output only the caption text."      -> the user turn, as a JSON string literal
  "You are a music annotation assistant for the ACE-Step 1.5 music generation model, producing training captions that conform to the ACE-Step 1.5XL dataset annotation schema. Output ONLY the caption text \u2014 no preamble, no headings, no markdown.\n\nCAPTION SCHEMA (exact shape):\n[Trigger Tag], [Primary Genre], [Subgenre/Mood], [2-3 Specific Instruments], [Vocal Style], [Production & Mix], [Era/Aesthetic]. [2-3 sentences detailing dynamic build, arrangement transitions, and energy flow].\n\nRULES:\n1. FRONT-LOAD the conditioning keywords: 5-12 comma-separated keywords (max 15) at the very start. Never open with a sentence.\n2. Use CONCRETE instruments and gear (e.g. Vox Continental organ, Gibson SG with P-90 pickups, Fender Rhodes bass, 808 sub bass, gated reverb drums).\n3. ALWAYS declare vocal presence and character: male vocal, female vocal, male baritone, raspy vocals, powerful belting, clean vocals, screamed vocals, whispered vocals, choir, or instrumental / no vocals.\n4. Follow the tag list with 2-3 sentences describing how the energy moves through the track (dynamic build, arrangement transitions, energy flow).\n5. NEVER put BPM, key, or time signature in the caption \u2014 those are dedicated numeric/categorical metadata fields.\n6. No headings, no field labels, no bullet points, no markdown. Output ONLY the caption text.\n7. Do not transcribe, quote, or repeat lyrics \u2014 describe the vocal delivery instead. Never repeat a word or phrase for emphasis.\n8. Keep the WHOLE caption to the keywords plus 2-3 sentences: about 70-90 words, and NEVER more than 120. A long essay is not a caption.\n\nGOLD EXAMPLE (conforms \u2014 imitate this shape):\nheavy metal, doom metal, doom-laden, ominous, downtuned guitar, distorted bass, thunderous drums, nasal male vocal, raw 1970s analog production, heavy low-end. Opens with a slow, ominous riff before building into a plaintive verse with wailing vocals. Reaches a crushing peak and fades into a bleak, atmospheric outro.\n\nWORTHLESS EXAMPLE (violates the schema \u2014 never produce this):\nGenre & Style: A high-energy Garage Rock track with a raw, lo-fi aesthetic. Tempo: Fast, driving tempo at approximately 120 BPM. Vocals & Lyrics: ... the repeated refrain: \"Baby, don't go, don't go, don't go, don't go, don't go, don't go...\""       -> the ACE-Step 1.5XL annotation schema (system turn),
                             as a JSON string literal. Appended to the model's
                             own identity line, never substituted for it.
  512      -> int (Concise Tags ~64, else ~512)
  "acdc"          -> trigger tag as a JSON string literal
  1.15  -> float, 1.0 = off
  6     -> int, 0 = off
"""
import os
import sys
import json
import glob
import tempfile
import subprocess
from pathlib import Path

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"


def _install():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "accelerate", "huggingface_hub", "hf-transfer",
                    "soundfile", "librosa", "numba", "tinytag", "tqdm"],
                   check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/huggingface/transformers@v4.51.3-Qwen2.5-Omni-preview",
                    "qwen-omni-utils[decord]"],
                   check=False)


_install()

import torch  # noqa: E402
from transformers import Qwen2_5OmniForConditionalGeneration, Qwen2_5OmniProcessor  # noqa: E402
from qwen_omni_utils import process_mm_info  # noqa: E402

AUDIO_FOLDER = "/kaggle/input/acestep-audio"
CAPTION_PROMPT = "Annotate this audio clip as ONE caption in the exact schema above. Output only the caption text."
SYSTEM_PROMPT = "You are a music annotation assistant for the ACE-Step 1.5 music generation model, producing training captions that conform to the ACE-Step 1.5XL dataset annotation schema. Output ONLY the caption text \u2014 no preamble, no headings, no markdown.\n\nCAPTION SCHEMA (exact shape):\n[Trigger Tag], [Primary Genre], [Subgenre/Mood], [2-3 Specific Instruments], [Vocal Style], [Production & Mix], [Era/Aesthetic]. [2-3 sentences detailing dynamic build, arrangement transitions, and energy flow].\n\nRULES:\n1. FRONT-LOAD the conditioning keywords: 5-12 comma-separated keywords (max 15) at the very start. Never open with a sentence.\n2. Use CONCRETE instruments and gear (e.g. Vox Continental organ, Gibson SG with P-90 pickups, Fender Rhodes bass, 808 sub bass, gated reverb drums).\n3. ALWAYS declare vocal presence and character: male vocal, female vocal, male baritone, raspy vocals, powerful belting, clean vocals, screamed vocals, whispered vocals, choir, or instrumental / no vocals.\n4. Follow the tag list with 2-3 sentences describing how the energy moves through the track (dynamic build, arrangement transitions, energy flow).\n5. NEVER put BPM, key, or time signature in the caption \u2014 those are dedicated numeric/categorical metadata fields.\n6. No headings, no field labels, no bullet points, no markdown. Output ONLY the caption text.\n7. Do not transcribe, quote, or repeat lyrics \u2014 describe the vocal delivery instead. Never repeat a word or phrase for emphasis.\n8. Keep the WHOLE caption to the keywords plus 2-3 sentences: about 70-90 words, and NEVER more than 120. A long essay is not a caption.\n\nGOLD EXAMPLE (conforms \u2014 imitate this shape):\nheavy metal, doom metal, doom-laden, ominous, downtuned guitar, distorted bass, thunderous drums, nasal male vocal, raw 1970s analog production, heavy low-end. Opens with a slow, ominous riff before building into a plaintive verse with wailing vocals. Reaches a crushing peak and fades into a bleak, atmospheric outro.\n\nWORTHLESS EXAMPLE (violates the schema \u2014 never produce this):\nGenre & Style: A high-energy Garage Rock track with a raw, lo-fi aesthetic. Tempo: Fast, driving tempo at approximately 120 BPM. Vocals & Lyrics: ... the repeated refrain: \"Baby, don't go, don't go, don't go, don't go, don't go, don't go...\""
MAX_NEW_TOKENS = 512
BATCH_SIZE = 1
CUSTOM_TAG = "acdc"
REPETITION_PENALTY = 1.15
NO_REPEAT_NGRAM = 6

# The model's own identity line, kept verbatim (set in stone). The ACE-Step
# annotation SCHEMA in SYSTEM_PROMPT is APPENDED to it, never substituted for it.
QWEN_IDENTITY = (
    "You are Qwen, a virtual human developed by the Qwen Team, Alibaba Group, "
    "capable of perceiving auditory and visual inputs, as well as generating "
    "text and speech."
)
SUPPORTED_FORMATS = {'.wav', '.mp3', '.flac', '.m4a', '.ogg', '.aac', '.wma'}

# ---- AUTO-RESOLVED AUDIO FOLDER -------------------------------------------------
# Kaggle mounts inputs at /kaggle/input/<slug>, and the slug depends on how the
# dataset was attached. If the path above does not exist, use the first input
# directory that actually contains audio files.
if not os.path.isdir(AUDIO_FOLDER):
    for _root, _dirs, _files in os.walk("/kaggle/input"):
        if any(os.path.splitext(_f)[1].lower() in SUPPORTED_FORMATS for _f in _files):
            AUDIO_FOLDER = _root
            break
print("AUDIO_FOLDER =", AUDIO_FOLDER)
# ---------------------------------------------------------------------------------


def is_valid_json(path):
    try:
        json.load(open(path))
        return True
    except Exception:
        return False


# ---- Model source: prefer a cached weights dataset under /kaggle/input ----
MODEL_SOURCE = None
for candidate in sorted(glob.glob("/kaggle/input/**/config.json", recursive=True)):
    d = os.path.dirname(candidate)
    if is_valid_json(candidate) and any(
        f.endswith((".safetensors", ".bin", ".pt")) for f in os.listdir(d)
    ):
        MODEL_SOURCE = d
        break

if MODEL_SOURCE is None:
    MODEL_SOURCE = "ACE-Step/acestep-captioner"
    hf = os.environ.get("HF_TOKEN")
    if hf:
        from huggingface_hub import login
        login(token=hf, add_to_git_credential=False)

torch_dtype = torch.float16
load_kwargs = {
    "device_map": "balanced",
    "max_memory": {0: "10GiB", 1: "10GiB"},
    "offload_folder": "/kaggle/working/offload",
    "trust_remote_code": True,
    "torch_dtype": torch_dtype,
}
try:
    import flash_attn  # noqa: F401
    load_kwargs["attn_implementation"] = "flash_attention_2"
except ImportError:
    load_kwargs["attn_implementation"] = "sdpa"

model = Qwen2_5OmniForConditionalGeneration.from_pretrained(MODEL_SOURCE, **load_kwargs)
model.disable_talker()
processor = Qwen2_5OmniProcessor.from_pretrained(MODEL_SOURCE, trust_remote_code=True)

audio_files = sorted(
    p for p in Path(AUDIO_FOLDER).rglob("*")
    if p.suffix.lower() in SUPPORTED_FORMATS and p.is_file()
)


def truncate_audio(audio_path, max_seconds=120):
    if not max_seconds or max_seconds <= 0:
        return audio_path
    import librosa  # noqa: PLC0415
    import soundfile as sf  # noqa: PLC0415
    try:
        y, sr = librosa.load(audio_path, sr=None, mono=False, duration=max_seconds)
        ext = os.path.splitext(audio_path)[1]
        fd, tmp = tempfile.mkstemp(suffix=ext)
        os.close(fd)
        if y.ndim > 1:
            sf.write(tmp, y.T, sr)
        else:
            sf.write(tmp, y, sr)
        return tmp
    except Exception:
        return audio_path


def extract_reply(text):
    if "assistant\\n" in text:
        return text.split("assistant\\n")[-1].strip()
    if "assistant" in text:
        return text.split("assistant")[-1].strip()
    return text.strip()


results = []
for i in range(0, len(audio_files), BATCH_SIZE):
    batch = audio_files[i:i + BATCH_SIZE]
    try:
        truncated = []
        for f in batch:
            truncated.append(truncate_audio(str(f)))

        conversations = [
            [
                {"role": "system", "content": [{"type": "text", "text": (
                    QWEN_IDENTITY + "\n\n" + SYSTEM_PROMPT)}]},
                {"role": "user", "content": [
                    {"type": "audio", "audio": t},
                    {"type": "text", "text": CAPTION_PROMPT},
                ]},
            ]
            for t in truncated
        ]
        text_input = processor.apply_chat_template(
            conversations, add_generation_prompt=True, tokenize=False
        )
        audios, images, videos = process_mm_info(conversations, use_audio_in_video=False)
        inputs = processor(
            text=text_input, audio=audios, images=images, videos=videos,
            return_tensors="pt", padding=True, use_audio_in_video=False,
        ).to(model.device).to(model.dtype)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs, use_audio_in_video=False, return_audio=False,
                max_new_tokens=MAX_NEW_TOKENS,
                # Greedy decoding with no penalty is what let a caption loop on a
                # repeated lyric phrase ~200 times until the token cap.
                repetition_penalty=REPETITION_PENALTY,
                no_repeat_ngram_size=NO_REPEAT_NGRAM,
            )
        full_texts = processor.batch_decode(
            output_ids, skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )
        for f, t, ft in zip(batch, truncated, full_texts):
            caption = extract_reply(ft)
            if CUSTOM_TAG:
                caption = f"{CUSTOM_TAG}, {caption}"
            results.append({"file": f.name, "caption": caption})
            print("OK", f.name, caption[:100])
            if t != str(f) and t.startswith(tempfile.gettempdir()):
                try:
                    os.remove(t)
                except Exception:
                    pass
    except Exception as e:
        import traceback  # noqa: PLC0415
        traceback.print_exc()
        for f in batch:
            results.append({"file": f.name, "caption": f"ERROR: {e}"})

with open("/kaggle/working/captions_out.json", "w") as out_f:
    json.dump({"results": results}, out_f, indent=2)
print("DONE", len(results))